In [ ]:
##! pip install pandas-ta==0.2.23b
import math
import pdb
from fyers_apiv3 import fyersModel
import pyotp
import pandas as pd
import requests as req
import datetime as dt
from urllib.parse import parse_qs, urlparse
import time
import talib as ta
import warnings
import joblib
from time import sleep
warnings.filterwarnings("ignore")
import numpy as np
import glob

In [ ]:
data_source = r"F:\Cources\Office\VSINTERN\Histoty\5"
path = glob.glob(f"{data_source}\*.csv")[:2]
path

In [ ]:
data = [pd.read_csv(i) for i in path] if type(path) == list else pd.read_csv(path)
data.head()

In [ ]:
def indicator(data):
    """
    Code your indicator logic here
    
    
    Should:
        return list of dataframe if data passed is list of dataframe. assuming that data is of different instrument. 
    """
    
    return data
indicator_data  = [indicator(i) for i in data] if type(data) == list else indicator(data)

In [ ]:
def signal_generator(data):
    """
    Code your strategy scaner aand signal generator logic here
    
    
    Should:
        return list of dataframe if data passed is list of dataframe. assuming that data is of different instrument. 
    """
    return data

generated_signal  = [signal_generator(i) for i in indicator_data] if type(indicator_data) == list else signal_generator(indicator_data)

In [ ]:
def consolidate_data(data_list):
    """
    consolidate each data in data_list
    
    """
    if type(data_list) == list:
        data = pd.concat(data_list)
        data['Date'] = pd.to_datetime(data['Date'])
        data.sort_values(by='Date', inplace=True)
        return data
    else:
        return data_list

consolidated_data = consolidate_data(indicator_data)


In [ ]:
class Backtesting_Engine:
    def __init__(self, data=None, initial_capital=1000000, lot_size=1, sl=0.02, tgt=0.04):
        self.data = data
        self.initial_capital = initial_capital
        self.lot_size = lot_size
        self.position = {}
        columns=['Entry Date', 'Symbol', "Entry_Price",
                 "SL","TGT","Quantity","Exit Date","Exit_Price", 
                 "Brokerage","STT","Stamp Duty","SEBI Charges","Transaction Cost","GST","Total Cost","Net P&L"
                 ]
        self.ledger = pd.DataFrame(columns=columns)
        self.__brokerages__()
        

    def __brokerages__(self):
        """
        Trading charges are taken from Zerodha's brokerage Charges portal
        url: https://zerodha.com/charges/#tab-equities
        """
        self.SEBI = 0.0001/100 #on both side
        self.Transaction_cost = 0.00297/100 #on both side
        self.brokerage = 0.03/100 #On both side
        self.GST = 18/100 #on brokerage, SEBI, Transaction cost
        self.STT = 0.025/100 #on both side
        
        
        self.stamp = 0.003/100 #on buy side


    def transaction_cost_caluclation(self, price, type_of_trade, quantity):
        """ArithmeticError
        Return : array as >> [SEBI,TC, Brokerage, GST, STT, Stamp Duty, Total Cost]
        """
        turnover = price * quantity
        #print(f"Turnover: {turnover}")
        #Cost FACtor
        cost_factor = np.array([self.SEBI, self.Transaction_cost])
        #Cost of SEBU, Transaction cost and Brokerage
        cost = turnover * cost_factor
        #Brokrage cost with a cap of 30
        cost = np.append(cost, np.min([self.brokerage * turnover,30]))
        #GST on brokerage|SEBI|Transaction cost
        cost = np.append(cost, self.GST * cost.sum())
        #STT cost
        cost = np.append(cost, self.STT * turnover)
        #Stamp duty
        if type_of_trade == 'buy':
            cost = np.append(cost, self.stamp * turnover)
        else:
            cost = np.append(cost, 0)
        cost = np.append(cost, cost.sum())
        return cost
        
    

BE = Backtesting_Engine()
list(BE.transaction_cost_caluclation(1000, 'buy', 100000))

In [ ]:
#pd.DataFrame(cost, index=['Brokerage','STT','Stamp Duty','Total Cost'], columns=['Cost'])

In [ ]:
capital = 100000
#{Symobl:[Entry Date, Entry_Price, SL, TGT, Quantity, Exit Date, Exit_Price,Costs, Net P&L]}

columns  =['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount","Y"]
position = {}
ledger = pd.DataFrame(columns=columns)
for index, row in consolidated_data.iterrows():
    symb = row['Symbol']
    entry_price = row['Close']
    signal = row['signal']
    _date_ = row['Date']
    rtp = 0.01
    if symb not in position:
        loss_per_trade = capital*rtp
        if signal == 1:
            sl = entry_price * (1-0.02)
            tgt = entry_price * (1+0.04)
            
            qty = math.floor(loss_per_trade/abs(entry_price-sl))
            if qty>0:
                cost = BE.transaction_cost_caluclation(entry_price, 'buy', qty)
                amuont = cost[-1] + entry_price*qty
                position[symb] = [_date_,symb, entry_price,qty, sl, tgt, 'buy', cost,-1*amuont,"Entry"]
                ledger.loc[len(ledger)] = position[symb]
                capital -= (cost[-1]+entry_price*qty)
        elif signal == -1:
            sl = entry_price * (1+0.02)
            tgt = entry_price * (1-0.04)
            
            qty = math.floor(loss_per_trade/abs(entry_price-sl))
            if qty>0:
                cost = BE.transaction_cost_caluclation(entry_price, 'sell', qty)
                amuont = cost[-1] + entry_price*qty
                position[symb] = [_date_,symb, entry_price,qty, sl, tgt, 'sell', cost,amuont,"Entry"]
                ledger.loc[len(ledger)] = position[symb]
                capital += (-cost[-1]+entry_price*qty)
    elif symb in position:
        position_data = position[symb]
        type_ = position_data[6]
        qty = position_data[3]
        if type_ == 'buy':
            if position_data[4] >= entry_price:
                #SL Hit
                #['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount"]
                cost = BE.transaction_cost_caluclation(entry_price, 'sell', qty)
                capital-= cost[-1]
                capital += entry_price*qty
                amuont = entry_price*qty - cost[-1]
                ledger.loc[len(ledger)] = [_date_,symb,entry_price,qty,0,0,"sell",cost,amuont,"SL"]
                del position[symb]
            elif position_data[5] <= entry_price:
                cost = BE.transaction_cost_caluclation(entry_price, 'sell', qty)
                capital-= cost[-1]
                capital += entry_price*qty
                amuont = entry_price*qty - cost[-1]
                ledger.loc[len(ledger)] = [_date_,symb,entry_price,qty,0,0,"sell",cost,amuont,"TGT"]
                del position[symb]
        elif type_ == 'sell':
            if position_data[4] <= entry_price:
                #SL Hit
                #['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount"]
                cost = BE.transaction_cost_caluclation(entry_price, 'buy', qty)
                capital-= cost[-1]
                capital -= entry_price*qty
                amuont = entry_price*qty - cost[-1]
                ledger.loc[len(ledger)] = [_date_,symb,entry_price,qty,0,0,"buy",cost,-amuont,"SL"]
                del position[symb]
            elif position_data[5] >= entry_price:
                cost = BE.transaction_cost_caluclation(entry_price, 'buy', qty)
                capital-= cost[-1]
                capital -= entry_price*qty
                amuont = entry_price*qty - cost[-1]
                ledger.loc[len(ledger)] = [_date_,symb,entry_price,qty,0,0,"buy",cost,-amuont,"TGT"]
                del position[symb]


In [ ]:
#cost_columns = ["SEBI","TC","Brokerage","GST","STT","Stamp Duty","Total Cost"]
#data = pd.DataFrame(ledger['Cost'].tolist(), columns=cost_columns, index=ledger['Entry Date'])    

In [ ]:
copy_ledger = ledger.copy()
#copy_ledger.set_index('Entry Date', inplace=True)
copy_ledger

In [ ]:
copy_ledger.sort_values("Date",inplace=True)#['Amount'].cumsum().plot()
copy_ledger['Date'] = pd.to_datetime(copy_ledger['Date'])
copy_ledger.sort_values("Date")

In [ ]:
counts_ = copy_ledger['Y'].value_counts()
#counts_['SL']/(counts_.sum()/2), counts_['TGT']/(counts_.sum()/2)
counts_

In [ ]:
import plotly.express as px
copy_ledger['Curve'] = copy_ledger.Amount.cumsum()+capital
fig=  px.line(copy_ledger, x="Date",y="Curve")
fig.show()

In [ ]:
copy_ledger.to_csv("Ledger.csv", index=False)

In [ ]:
data = pd.DataFrame(
    {"A":[54,654,65432,872,],
     "B":[85,85,997,63,]}
)

In [ ]:
data['A2'] = data['A'].shift(1)

In [ ]:
data

In [ ]:
capital = 100000
#{Symobl:[Entry Date, Entry_Price, SL, TGT, Quantity, Exit Date, Exit_Price,Costs, Net P&L]}

columns  =['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount","Y"]
position = {}
ledger = pd.DataFrame(columns=columns)
for index, row in consolidated_data.iterrows():
    symb = row['Symbol']
    entry_price = row['Close']
    signal = row['signal']
    _date_ = row['Date']

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_trading_chart(df):
    """
    df must contain:
    ['Open','High','Low','Close','Volume','EWM','Cap','Floor']
    Index must be datetime.
    """

    # Create subplots (price + volume)
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.75, 0.25]
    )

    # ======================
    # 1️⃣ Candlestick Chart
    # ======================
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            name="OHLC"
        ),
        row=1, col=1
    )

    # ======================
    # 2️⃣ Indicator Lines
    # ======================

    # EWM
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['EWM'],
            mode='lines',
            name='EWM',
            line=dict(width=1.5)
        ),
        row=1, col=1
    )

    # Cap
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['Cap'],
            mode='lines',
            name='Cap',
            line=dict(width=1.5)
        ),
        row=1, col=1
    )

    # Floor
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['Floor'],
            mode='lines',
            name='Floor',
            line=dict(width=1.5)
        ),
        row=1, col=1
    )

    # ======================
    # 3️⃣ Volume
    # ======================
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df['Volume'],
            name='Volume'
        ),
        row=2, col=1
    )

    # ======================
    # Layout
    # ======================
    fig.update_layout(
        title="OHLCV with EWM, Cap & Floor",
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        height=800,
        legend=dict(orientation="h")
    )

    fig.show()

plot_trading_chart(indicator_data.tail(200))